In [1]:
# =========================================================
# TASK 6: MULTI-MODEL COMPARISON BENCHMARK
# =========================================================

import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from imblearn.over_sampling import SMOTE
from skeLCS import eLCS

# Import 3 Conventional Machine Learning Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# Import evaluation metrics
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# 1. LOAD AND INITIALIZE RAW DATASET PIPELINES
df = pd.read_csv('ardd_fatalities.csv', low_memory=False)
if 'Unnamed: 23' in df.columns:
    df = df.drop(columns=['Unnamed: 23'])

# Isolate target criteria
df['Age'] = df['Age'].replace([-9, '-9', '-9.0', 'Unknown'], np.nan)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df = df.dropna(subset=['Age'])
df = df[df['Age'] >= 17].copy()
df['Target_Youth'] = (df['Age'] <= 25).astype(int)

feature_cols = [
    'State', 'Speed Limit', 'Crash Type', 'Bus Involvement', 
    'Heavy Rigid Truck Involvement', 'Articulated Truck Involvement', 
    'Dayweek', 'Time of day'
]

X_raw_df = df[feature_cols].copy()
y = df['Target_Youth'].values

# Split data first to guarantee zero test leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw_df, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------
# PREPARATION FOR CONFIGURATION 1 (Raw/Minimal LCS Mapping)
# ---------------------------------------------------
X_train_minimal = X_train_raw.copy()
X_test_minimal = X_test_raw.copy()
for col in X_train_minimal.columns:
    X_train_minimal[col] = X_train_minimal[col].astype(str).astype('category').cat.codes
    X_test_minimal[col] = X_test_minimal[col].astype(str).astype('category').cat.codes

# ---------------------------------------------------
# PREPARATION FOR CONFIGURATIONS 2-6 (Advanced Engineering + Imputation)
# ---------------------------------------------------
X_tr_eng = X_train_raw.copy()
X_te_eng = X_test_raw.copy()

# Apply the Feature Engineering step from Task 3
for target_df in [X_tr_eng, X_te_eng]:
    target_df['Time_Clean'] = pd.to_datetime(target_df['Time of day'], format='%H:%M:%S', errors='coerce').dt.hour
    target_df['Is_Night'] = target_df['Time_Clean'].apply(lambda x: 1 if (x >= 22 or x <= 5) else 0)
    target_df.drop(columns=['Time_Clean'], inplace=True)

cat_cols = X_tr_eng.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_tr_eng.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Fit Imputers & Encoders only on Training Splits
imp_cat = SimpleImputer(strategy='most_frequent')
X_tr_eng[cat_cols] = imp_cat.fit_transform(X_tr_eng[cat_cols])
X_te_eng[cat_cols] = imp_cat.transform(X_te_eng[cat_cols])

ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_tr_eng[cat_cols] = ord_enc.fit_transform(X_tr_eng[cat_cols])
X_te_eng[cat_cols] = ord_enc.transform(X_te_eng[cat_cols])

if num_cols:
    imp_num = SimpleImputer(strategy='median')
    X_tr_eng[num_cols] = imp_num.fit_transform(X_tr_eng[num_cols])
    X_te_eng[num_cols] = imp_num.transform(X_te_eng[num_cols])

X_train_clean = X_tr_eng.values.astype(np.float32)
X_test_clean = X_te_eng.values.astype(np.float32)

# Create the SMOTE Balanced dataset variation
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_clean, y_train)

# ---------------------------------------------------
# MODEL EXECUTION AND BENCHMARK ENGINE
# ---------------------------------------------------
results = {}

def evaluate_model(model_instance, train_X, train_y, test_X, model_label):
    print(f"Training {model_label}...")
    start = time.time()
    model_instance.fit(train_X, train_y)
    t_time = time.time() - start
    preds = model_instance.predict(test_X)
    
    results[model_label] = {
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, preds, average='weighted'),
        'F1-Score': f1_score(y_test, preds, average='weighted'),
        'Train Time (s)': t_time
    }

# 1. Original eLCS on Raw Data Split
evaluate_model(eLCS(learning_iterations=5000, N=500, p_spec=0.5), 
               X_train_minimal.values, y_train, X_test_minimal.values, '1. Original LCS (Raw)')

# 2. Original eLCS on Clean Data Split (Without SMOTE)
evaluate_model(eLCS(learning_iterations=5000, N=500, p_spec=0.5), 
               X_train_clean, y_train, X_test_clean, '2. Original LCS (Preprocessed)')

# 3. Proposed Improved LCS System (Clean Data + SMOTE Balance)
evaluate_model(eLCS(learning_iterations=5000, N=500, p_spec=0.5), 
               X_train_resampled, y_train_resampled, X_test_clean, '3. Proposed Improved LCS')

# 4. Conventional Model: Random Forest
evaluate_model(RandomForestClassifier(random_state=42, n_estimators=100), 
               X_train_clean, y_train, X_test_clean, '4. Random Forest')

# 5. Conventional Model: Decision Tree
evaluate_model(DecisionTreeClassifier(random_state=42), 
               X_train_clean, y_train, X_test_clean, '5. Decision Tree')

# 6. Conventional Model: Logistic Regression
evaluate_model(LogisticRegression(random_state=42, max_iter=1000), 
               X_train_clean, y_train, X_test_clean, '6. Logistic Regression')

# ---------------------------------------------------
# PRINT SUMMARY COMPARISON MATRIX
# ---------------------------------------------------
summary_df = pd.DataFrame(results).T
print("\n" + "="*80)
print("             TASK 6: FINAL MODEL COMPARISON BENCHMARK")
print("="*80)
print(summary_df.to_string(formatters={
    'Accuracy': '{:,.4f}'.format, 'Precision': '{:,.4f}'.format,
    'Recall': '{:,.4f}'.format, 'F1-Score': '{:,.4f}'.format, 'Train Time (s)': '{:,.2f}'.format
}))
print("="*80)


C:\Users\ASUS F15\AppData\Local\Temp\ipykernel_69452\2178419746.py:69: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_tr_eng.select_dtypes(include=['object', 'category']).columns.tolist()


Training 1. Original LCS (Raw)...
Training 2. Original LCS (Preprocessed)...
Training 3. Proposed Improved LCS...
Training 4. Random Forest...
Training 5. Decision Tree...
Training 6. Logistic Regression...

             TASK 6: FINAL MODEL COMPARISON BENCHMARK
                               Accuracy Precision Recall F1-Score Train Time (s)
1. Original LCS (Raw)            0.7207    0.5194 0.7207   0.6037           2.63
2. Original LCS (Preprocessed)   0.7207    0.5194 0.7207   0.6037           3.51
3. Proposed Improved LCS         0.6008    0.6890 0.6008   0.6213           5.26
4. Random Forest                 0.7070    0.6505 0.7070   0.6519           1.77
5. Decision Tree                 0.7069    0.6474 0.7069   0.6481           0.04
6. Logistic Regression           0.7207    0.5194 0.7207   0.6037           0.11
